# 第5回：何を、いつ、何のために予測するか

**今日の問い：モデル構築より前に決めるべきことは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 利用者・判断・予測時点を1文にする
- 目的変数と利用可能な説明変数を分け、リーク候補を自動監査する
- 誤りのコストから期待値を計算し、指標と閾値を業務要件で決める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 予測時点：モデルを実際に使う瞬間
- ベースライン：複雑なモデルと比較する単純な基準
- コスト行列：誤りの種類ごとの損失をまとめた表
- 期待コスト：確率×損失で見積もる平均的な損失
- リーク監査：目的変数と強く結びつく怪しい列を洗い出す確認

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 「良いモデル」の前に「正しい問い」

初心者がいちばん飛ばしがちで、実は最も効くのがこの回です。**どんなに精度が高くても、問いの立て方が
間違っていれば役に立ちません**。モデルを組む前に、次を1文で言えるようにします。

> **誰が・いつ・何を予測し・その結果をどう使うか。**

例：*実験条件を決める時点で使える情報から収率を予測し、優先して試す条件を選ぶ。*

ここで決定的に大事なのが**予測時点**です。「いつ予測するか」を決めると、その時点で**まだ手に入って
いない情報は使えない**と分かります。実験後にしか得られない値を入力に混ぜると、練習では高得点でも
本番でまったく使えない「ズル（リーク）」になります。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 計画時に使える列／使えない列を仕分ける

このデータで「実験条件を決める時点」を予測時点とすると、収率・活性・純度・測定後シグナルは
**まだ存在しません**。使える列と使えない列を、はっきり2つのリストに分けます。この仕分けが
特徴量選びの土台になります。


In [ ]:
available_at_planning = [
    "scaffold_group", "solvent", "catalyst", "temperature_c", "reaction_time_h",
    "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds",
]
unavailable_at_planning = ["yield_pct", "active", "post_assay_signal", "purity_pct"]
print("計画時に使える列:", available_at_planning)
print("実験後に得られる列:", unavailable_at_planning)


### 読みどころ

`unavailable_at_planning`の列は「結果」や「結果に強く連動する測定値」です。これらを特徴量に入れると
リークになります。**列の名前ではなく「その値がいつ確定するか」で判断する**のがコツです。


## TRY：まず「単純な基準（ベースライン）」を作る

複雑なモデルに進む前に、**平均値だけ／多数派だけ**を答える最も単純なモデルを作ります。これが
比較の出発点（ものさし）になります。以降のどのモデルも、まずこれを超えることが最低条件です。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=42)
reg = DummyRegressor(strategy="mean").fit(train[["molecular_weight"]], train["yield_pct"])
cls = DummyClassifier(strategy="most_frequent").fit(train[["molecular_weight"]], train["active"])
print("平均収率だけのMAE:", round(mean_absolute_error(valid["yield_pct"], reg.predict(valid[["molecular_weight"]])), 2))
print("多数派だけの正解率:", round(accuracy_score(valid["active"], cls.predict(valid[["molecular_weight"]])), 3))


### 出力の読み方

- **MAE（平均絶対誤差）**：予測が平均どれだけ外れるか。単位は収率と同じ%。「平均値だけ」でこの誤差、というものさしです。
- **多数派だけの正解率**が高く出ることに驚くかもしれません。活性が少ないデータでは「全部を多数派と答える」だけで正解率が高くなります。**だから正解率は当てにならない**——第8回でF1を学ぶ動機になります。
- 本命モデルは、この2つの数字を**はっきり上回って初めて価値がある**と考えます。


## CORE深掘り：リーク候補を自動で洗い出す

「どの列がリークか」を人手で全部見るのは大変です。目的変数と**極端に強く連動する列**は、結果由来の
情報が紛れている疑いがあります。それを見つける監査を関数にしておくと、自社データでも使い回せます。


In [ ]:
def leakage_audit(frame, target: str, threshold: float = 0.9):
    "目的変数と相関が極端に高い数値列を、リーク候補として洗い出す。"
    numeric = frame.select_dtypes(include="number")
    corr = numeric.corrwith(frame[target]).abs().drop(labels=[target], errors="ignore")
    report = corr.sort_values(ascending=False).to_frame("|相関|")
    report["リーク候補"] = report["|相関|"] >= threshold
    return report.round(3)

display(leakage_audit(df, target="active", threshold=0.6))
print("post_assay_signalは測定後の値。相関が高くても計画時には使えない。")


### 出力の読み方と注意

- `active`との|相関|が高い順に並びます。`post_assay_signal`が上位に来るはず——これは**活性測定後の値**なので、計画時には存在せず、使えばリークです。
- ただしこの監査は**あくまで補助**。相関が低くてもリークする列（例：実験日から結果を推測できる場合）もあります。最終判断は「その値がいつ確定するか」で人が行います。
- `threshold`はリーク候補とみなす相関の閾値。厳しく見たいなら下げます。


## TRY：自分のテーマを1枚に整理する

次の8点を、機密を書かずに埋めます。**利用者／判断／予測時点／目的変数／使える列／使えない列／
回帰か分類か／単純な基準**。埋まらない項目があれば、それが今いちばん詰めるべき点です。

## ASK COPILOT

Copilotには、曖昧な項目を勝手に埋めさせず「確認すべき質問」の形で返すよう頼みます。


## DEEP DIVE：最適な閾値は「コスト」で決まる

分類モデルは確率を出し、ある**閾値**を超えたら「活性」と判定します。既定の0.5が最適とは限りません。
最適な閾値は指標ではなく、**誤りのコスト**で決まります。ここでは「見逃し（偽陰性）＝有望条件を逃す損失」が
「偽陽性＝無駄な追試」より10倍高い状況を想定し、期待コストが最小になる閾値を探します。


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(df[feat], df["active"], test_size=0.3, random_state=42, stratify=df["active"])
clf = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

cost_fn, cost_fp = 10, 1  # 見逃し=有望条件を逃す損失、偽陽性=無駄な追試
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba >= t).astype(int)
    fp = int(((pred == 1) & (y_te == 0)).sum())
    fn = int(((pred == 0) & (y_te == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
cost_table = pd.DataFrame(rows)
best = cost_table.loc[cost_table["期待コスト"].idxmin(), "閾値"]
display(cost_table)
print("コスト最小の閾値:", best, " / 見逃しが高いほど閾値は下がる")


### 出力の読み方

- 閾値を下げると「活性」と判定する数が増え、**偽陰性（見逃し）は減るが偽陽性は増える**トレードオフが表で見えます。
- 見逃しのコストが高いので、**最適閾値は0.5より低め**に出るはずです。「とりあえず0.5」がいかに恣意的かが分かります。
- コストの比（10:1）を変えれば最適閾値も動きます。**閾値はモデルの外側で、目的に合わせて選ぶ**ものだと理解できます。


### まとめ：指標は意思決定から逆算する

見逃しを避けたい探索段階なら**recall**寄り、追試コストが高い絞り込み段階なら**precision**寄り。
「良いスコア」を追うのではなく、「この予測で何を決め、間違えると何を失うか」から指標と閾値を選びます。
これがデータサイエンスを"意味のある学び"にする芯です。


## APPENDIX（任意・追加演習）

問題設定まわりのコードをもう少し。90分の外の自習向けです。まず**単純基準（Dummy）を戦略ごとに
比較**し、「どのベースラインを土俵にするか」を意識します。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

tr, va = train_test_split(df, test_size=0.25, random_state=42)
print("=== 回帰の単純基準 ===")
for strat in ["mean", "median"]:
    d = DummyRegressor(strategy=strat).fit(tr[["molecular_weight"]], tr["yield_pct"])
    print(f"{strat:14s} MAE={mean_absolute_error(va['yield_pct'], d.predict(va[['molecular_weight']])):.2f}")
print("=== 分類の単純基準 ===")
for strat in ["most_frequent", "stratified", "uniform"]:
    d = DummyClassifier(strategy=strat, random_state=42).fit(tr[["molecular_weight"]], tr["active"])
    print(f"{strat:14s} accuracy={accuracy_score(va['active'], d.predict(va[['molecular_weight']])):.3f}")


### 出力の読み方

- 回帰は`mean`と`median`でMAEが少し違います。分布が歪んでいると`median`が有利なことも。
- 分類の`most_frequent`は正解率が高く見えますが、これは第8回で学ぶ「不均衡の罠」。`stratified`/`uniform`はランダムに近い基準です。
- 本命モデルは、**これらのうち最も手強い基準**を超えて初めて価値があります。


### 予測時点チェックを関数にする

第5回本編の「使える列／使えない列」を、候補リストから自動で仕分ける関数にします。自社データでも
使い回せる、実務的な安全装置です。


In [ ]:
def check_feature_timing(candidate_features, available_now, target):
    "特徴量候補を『使える/使えない(リーク)』に仕分ける。"
    rows = []
    for col in candidate_features:
        if col == target:
            verdict = "目的変数(使わない)"
        elif col in available_now:
            verdict = "使える"
        else:
            verdict = "使えない(予測時点で未確定)"
        rows.append({"列": col, "判定": verdict})
    return pd.DataFrame(rows)

check_feature_timing(
    ["temperature_c", "logp", "yield_pct", "post_assay_signal", "active"],
    available_now=available_at_planning,
    target="active",
)


### 出力の読み方

`yield_pct`や`post_assay_signal`が「使えない(予測時点で未確定)」と仕分けられます。列名を眺めるだけでなく、
**このチェックを通してから特徴量を確定する**運用にすれば、リークの多くを機械的に防げます。


### 期待値で「試すか否か」を決める

活性確率を予測できたとして、「その条件を追試すべきか」を**期待利益**で判断する簡単な例です。
確率×利益からコストを引いて、プラスなら試す。第5回・第8回のコスト最適閾値の考え方の土台です。


In [ ]:
import numpy as np

proba = np.array([0.10, 0.40, 0.60, 0.85])   # 各条件の活性確率（仮）
gain_if_active, cost_of_test = 100, 20
table = pd.DataFrame({"活性確率": proba})
table["期待利益"] = proba * gain_if_active - cost_of_test
table["試す?"] = table["期待利益"] > 0
table.round(1)


### 出力の読み方

期待利益がプラスの条件だけ「試す?=True」になります。ここでは損益分岐の確率は`cost/gain=0.2`。
つまり**活性確率20%以上なら試す**が最適で、これがそのまま判定閾値になります。「閾値0.5」が絶対でない
理由が、利益の式から自然に出てくることを確認してください。


## よくある誤り

- 入手できる列をすべて使う
- 目的変数が測定や運用で不安定
- 精度目標だけで利用方法とコストが決まっていない

## SELF-STUDY（任意・30〜60分）

- 自社テーマを機密情報なしで問題設定キャンバスへ落とす
- 偽陽性・偽陰性のコストを入れ、期待コスト最小の閾値を計算する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 誰が何を判断するモデルか
2. 予測時点で本当に得られる列はどれか
3. コスト行列から最適な閾値をどう求めるか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
